In [1]:
import pandas as pd
import numpy as np

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

print("LightGBM:", lgb.__version__)
print("XGBoost:", xgb.__version__)

LightGBM: 4.7.0
XGBoost: 3.4.1


In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Sample:", sample_submission.shape)

Train: (691369, 14)
Test: (296302, 13)
Sample: (296302, 2)


In [3]:
y = train["addicted_label"]

X = train.drop(
    columns=["addicted_label", "id"]
)

X_test = test.drop(
    columns=["id"]
)

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

X: (691369, 12)
y: (691369,)
X_test: (296302, 12)


In [4]:
categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['gender', 'stress_level', 'academic_work_impact']


C:\Users\ASUS\AppData\Local\Temp\ipykernel_16592\986657640.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [5]:
for col in categorical_columns:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

In [6]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(
    skf.split(X, y)
)

print("Number of folds:", len(folds))

Number of folds: 5


In [7]:
lgb_oof = np.zeros(len(X))
xgb_oof = np.zeros(len(X))
cat_oof = np.zeros(len(X))

lgb_test = np.zeros(len(X_test))
xgb_test = np.zeros(len(X_test))
cat_test = np.zeros(len(X_test))

In [8]:
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",

    "n_estimators": 5000,
    "learning_rate": 0.03,

    "num_leaves": 63,
    "max_depth": -1,

    "min_child_samples": 30,

    "subsample": 0.85,
    "colsample_bytree": 0.85,

    "reg_alpha": 0.1,
    "reg_lambda": 1.0,

    "random_state": 42,

    "n_jobs": -1,
    "verbosity": -1
}

In [9]:
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",

    "n_estimators": 5000,
    "learning_rate": 0.03,

    "max_depth": 8,
    "min_child_weight": 5,

    "subsample": 0.85,
    "colsample_bytree": 0.85,

    "reg_alpha": 0.1,
    "reg_lambda": 1.0,

    "tree_method": "hist",

    "random_state": 42,

    "n_jobs": -1
}

In [10]:
cat_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",

    "iterations": 5000,
    "learning_rate": 0.03,

    "depth": 8,

    "l2_leaf_reg": 5,

    "random_seed": 42,

    "verbose": 100,

    "thread_count": -1,

    "allow_writing_files": False
}

In [11]:
cat_feature_indices = [
    X.columns.get_loc(col)
    for col in categorical_columns
]

print(cat_feature_indices)

[9, 10, 11]


In [12]:
X_cat = X.copy()
X_test_cat = X_test.copy()

In [14]:
X_cat = X.copy()
X_test_cat = X_test.copy()

for col in categorical_columns:
    X_cat[col] = X_cat[col].astype("object")
    X_test_cat[col] = X_test_cat[col].astype("object")

    X_cat[col] = X_cat[col].fillna("__MISSING__")
    X_test_cat[col] = X_test_cat[col].fillna("__MISSING__")

    X_cat[col] = X_cat[col].astype(str)
    X_test_cat[col] = X_test_cat[col].astype(str)

In [15]:
X_xgb = X.copy()
X_test_xgb = X_test.copy()

for col in categorical_columns:

    combined = pd.concat(
        [
            X_xgb[col],
            X_test_xgb[col]
        ],
        axis=0
    ).astype(str)

    categories = pd.Categorical(
        combined
    ).categories

    X_xgb[col] = pd.Categorical(
        X_xgb[col].astype(str),
        categories=categories
    ).codes

    X_test_xgb[col] = pd.Categorical(
        X_test_xgb[col].astype(str),
        categories=categories
    ).codes

In [16]:
cat_feature_indices = [
    X_cat.columns.get_loc(col)
    for col in categorical_columns
]

print(cat_feature_indices)

[9, 10, 11]


In [17]:
for col in categorical_columns:
    print(
        col,
        "missing:",
        X_cat[col].isna().sum()
    )

gender missing: 0
stress_level missing: 0
academic_work_impact missing: 0


In [22]:
for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print("\n")
    print("=" * 70)
    print(f"FOLD {fold}/5")
    print("=" * 70)


    # ========================================================
    # LIGHTGBM DATA
    # ========================================================

    X_train_lgb = X.iloc[train_idx]
    X_valid_lgb = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]


    # ========================================================
    # XGBOOST DATA
    # ========================================================

    X_train_xgb = X_xgb.iloc[train_idx]
    X_valid_xgb = X_xgb.iloc[valid_idx]


    # ========================================================
    # CATBOOST DATA
    # ========================================================

    X_train_cat = X_cat.iloc[train_idx]
    X_valid_cat = X_cat.iloc[valid_idx]


    # ========================================================
    # LIGHTGBM
    # ========================================================

    print("\n----- LIGHTGBM -----")

    lgb_model = lgb.LGBMClassifier(
        **lgb_params
    )

    lgb_model.fit(
        X_train_lgb,
        y_train,

        eval_set=[
            (X_valid_lgb, y_valid)
        ],

        eval_metric="auc",

        categorical_feature=categorical_columns,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=200,
                verbose=True
            ),
            lgb.log_evaluation(
                period=100
            )
        ]
    )

    lgb_valid_pred = lgb_model.predict_proba(
        X_valid_lgb
    )[:, 1]

    lgb_test_pred = lgb_model.predict_proba(
        X_test
    )[:, 1]

    lgb_oof[valid_idx] = lgb_valid_pred

    lgb_test += (
        lgb_test_pred / len(folds)
    )

    print(
        f"LightGBM Fold {fold} AUC: "
        f"{roc_auc_score(y_valid, lgb_valid_pred):.6f}"
    )


    # ========================================================
    # XGBOOST
    # ========================================================

    print("\n----- XGBOOST -----")

    xgb_model = xgb.XGBClassifier(
        **xgb_params
    )

    xgb_model.fit(
        X_train_xgb,
        y_train,

        eval_set=[
            (X_valid_xgb, y_valid)
        ],

        verbose=100
    )

    xgb_valid_pred = xgb_model.predict_proba(
        X_valid_xgb
    )[:, 1]

    xgb_test_pred = xgb_model.predict_proba(
        X_test_xgb
    )[:, 1]

    xgb_oof[valid_idx] = xgb_valid_pred

    xgb_test += (
        xgb_test_pred / len(folds)
    )

    print(
        f"XGBoost Fold {fold} AUC: "
        f"{roc_auc_score(y_valid, xgb_valid_pred):.6f}"
    )


    # ========================================================
    # CATBOOST
    # ========================================================

    print("\n----- CATBOOST -----")

    cat_model = CatBoostClassifier(
        **cat_params
    )

    cat_model.fit(
        X_train_cat,
        y_train,

        eval_set=(
            X_valid_cat,
            y_valid
        ),

        cat_features=cat_feature_indices,

        early_stopping_rounds=200
    )

    cat_valid_pred = cat_model.predict_proba(
        X_valid_cat
    )[:, 1]

    cat_test_pred = cat_model.predict_proba(
        X_test_cat
    )[:, 1]

    cat_oof[valid_idx] = cat_valid_pred

    cat_test += (
        cat_test_pred / len(folds)
    )

    print(
        f"CatBoost Fold {fold} AUC: "
        f"{roc_auc_score(y_valid, cat_valid_pred):.6f}"
    )



FOLD 1/5

----- LIGHTGBM -----


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.94341
[200]	valid_0's auc: 0.954096
[300]	valid_0's auc: 0.957729
[400]	valid_0's auc: 0.959561
[500]	valid_0's auc: 0.960279
[600]	valid_0's auc: 0.960753
[700]	valid_0's auc: 0.961128
[800]	valid_0's auc: 0.961382
[900]	valid_0's auc: 0.961604
[1000]	valid_0's auc: 0.961776
[1100]	valid_0's auc: 0.961944
[1200]	valid_0's auc: 0.962124
[1300]	valid_0's auc: 0.96221
[1400]	valid_0's auc: 0.962314
[1500]	valid_0's auc: 0.962384
[1600]	valid_0's auc: 0.962458
[1700]	valid_0's auc: 0.962495
[1800]	valid_0's auc: 0.962522
[1900]	valid_0's auc: 0.962561
[2000]	valid_0's auc: 0.962602
[2100]	valid_0's auc: 0.962615
[2200]	valid_0's auc: 0.962622
[2300]	valid_0's auc: 0.962636
[2400]	valid_0's auc: 0.96266
[2500]	valid_0's auc: 0.962684
[2600]	valid_0's auc: 0.962701
[2700]	valid_0's auc: 0.962703
[2800]	valid_0's auc: 0.962722
[2900]	valid_0's auc: 0.962733
[3000]	valid_0's auc: 0.962744
[3100]	valid_0's auc

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.94369
[200]	valid_0's auc: 0.954156
[300]	valid_0's auc: 0.958285
[400]	valid_0's auc: 0.960039
[500]	valid_0's auc: 0.960886
[600]	valid_0's auc: 0.961416
[700]	valid_0's auc: 0.961732
[800]	valid_0's auc: 0.962009
[900]	valid_0's auc: 0.962233
[1000]	valid_0's auc: 0.962413
[1100]	valid_0's auc: 0.96262
[1200]	valid_0's auc: 0.962768
[1300]	valid_0's auc: 0.962887
[1400]	valid_0's auc: 0.962989
[1500]	valid_0's auc: 0.963077
[1600]	valid_0's auc: 0.96314
[1700]	valid_0's auc: 0.963215
[1800]	valid_0's auc: 0.963263
[1900]	valid_0's auc: 0.96333
[2000]	valid_0's auc: 0.963386
[2100]	valid_0's auc: 0.963419
[2200]	valid_0's auc: 0.963423
[2300]	valid_0's auc: 0.96345
[2400]	valid_0's auc: 0.963474
[2500]	valid_0's auc: 0.963493
[2600]	valid_0's auc: 0.963523
[2700]	valid_0's auc: 0.963557
[2800]	valid_0's auc: 0.963575
[2900]	valid_0's auc: 0.963595
[3000]	valid_0's auc: 0.963613
[3100]	valid_0's auc: 

...

Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.945219
[200]	valid_0's auc: 0.955435
[300]	valid_0's auc: 0.959004
[400]	valid_0's auc: 0.960607
[500]	valid_0's auc: 0.961482
[600]	valid_0's auc: 0.961964
[700]	valid_0's auc: 0.962334
[800]	valid_0's auc: 0.962625
[900]	valid_0's auc: 0.962817
[1000]	valid_0's auc: 0.962901
[1100]	valid_0's auc: 0.963035
[1200]	valid_0's auc: 0.963208
[1300]	valid_0's auc: 0.963362
[1400]	valid_0's auc: 0.963457
[1500]	valid_0's auc: 0.963548
[1600]	valid_0's auc: 0.96362
[1700]	valid_0's auc: 0.963673
[1800]	valid_0's auc: 0.963714
[1900]	valid_0's auc: 0.963787
[2000]	valid_0's auc: 0.96387
[2100]	valid_0's auc: 0.963897
[2200]	valid_0's auc: 0.963957
[2300]	valid_0's auc: 0.963997
[2400]	valid_0's auc: 0.964023
[2500]	valid_0's auc: 0.964045
[2600]	valid_0's auc: 0.964058
[2700]	valid_0's auc: 0.964058
[2800]	valid_0's auc: 0.964059
Early stopping, best iteration is:
[2622]	valid_0's auc: 0.964064
LightGBM Fold 3

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.945265
[200]	valid_0's auc: 0.955953
[300]	valid_0's auc: 0.959607
[400]	valid_0's auc: 0.96133
[500]	valid_0's auc: 0.962109
[600]	valid_0's auc: 0.962564
[700]	valid_0's auc: 0.962908
[800]	valid_0's auc: 0.96316
[900]	valid_0's auc: 0.963382
[1000]	valid_0's auc: 0.963522
[1100]	valid_0's auc: 0.963591
[1200]	valid_0's auc: 0.96372
[1300]	valid_0's auc: 0.963796
[1400]	valid_0's auc: 0.963876
[1500]	valid_0's auc: 0.963958
[1600]	valid_0's auc: 0.964024
[1700]	valid_0's auc: 0.964096
[1800]	valid_0's auc: 0.96414
[1900]	valid_0's auc: 0.964176
[2000]	valid_0's auc: 0.964211
[2100]	valid_0's auc: 0.964225
[2200]	valid_0's auc: 0.964269
[2300]	valid_0's auc: 0.964285
[2400]	valid_0's auc: 0.964299
[2500]	valid_0's auc: 0.96431
[2600]	valid_0's auc: 0.964318
[2700]	valid_0's auc: 0.964343
[2800]	valid_0's auc: 0.964342
[2900]	valid_0's auc: 0.964358
[3000]	valid_0's auc: 0.964369
[3100]	valid_0's auc: 

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.944532
[200]	valid_0's auc: 0.954727
[300]	valid_0's auc: 0.958341
[400]	valid_0's auc: 0.960145
[500]	valid_0's auc: 0.960951
[600]	valid_0's auc: 0.961499
[700]	valid_0's auc: 0.961789
[800]	valid_0's auc: 0.962044
[900]	valid_0's auc: 0.962246
[1000]	valid_0's auc: 0.96245
[1100]	valid_0's auc: 0.962544
[1200]	valid_0's auc: 0.96267
[1300]	valid_0's auc: 0.962774
[1400]	valid_0's auc: 0.962848
[1500]	valid_0's auc: 0.962921
[1600]	valid_0's auc: 0.963004
[1700]	valid_0's auc: 0.963087
[1800]	valid_0's auc: 0.963116
[1900]	valid_0's auc: 0.963154
[2000]	valid_0's auc: 0.96319
[2100]	valid_0's auc: 0.963225
[2200]	valid_0's auc: 0.96326
[2300]	valid_0's auc: 0.963292
[2400]	valid_0's auc: 0.963333
[2500]	valid_0's auc: 0.963351
[2600]	valid_0's auc: 0.963379
[2700]	valid_0's auc: 0.963395
[2800]	valid_0's auc: 0.963401
[2900]	valid_0's auc: 0.963419
[3000]	valid_0's auc: 0.963441
[3100]	valid_0's auc:

In [23]:
lgb_score = roc_auc_score(
    y,
    lgb_oof
)

print(
    f"LightGBM OOF ROC-AUC: {lgb_score:.6f}"
)

LightGBM OOF ROC-AUC: 0.963683


In [24]:
xgb_score = roc_auc_score(
    y,
    xgb_oof
)

print(
    f"XGBoost OOF ROC-AUC: {xgb_score:.6f}"
)

XGBoost OOF ROC-AUC: 0.963667


In [25]:
cat_score = roc_auc_score(
    y,
    cat_oof
)

print(
    f"CatBoost OOF ROC-AUC: {cat_score:.6f}"
)

CatBoost OOF ROC-AUC: 0.963402


In [26]:
results = pd.DataFrame({
    "Model": [
        "LightGBM",
        "XGBoost",
        "CatBoost"
    ],

    "OOF ROC-AUC": [
        lgb_score,
        xgb_score,
        cat_score
    ]
})

results.sort_values(
    "OOF ROC-AUC",
    ascending=False
)

,Model,OOF ROC-AUC
0,LightGBM,0.963683
1,XGBoost,0.963667
2,CatBoost,0.963402


In [27]:
blend_333 = (
    0.333333 * lgb_oof +
    0.333333 * xgb_oof +
    0.333334 * cat_oof
)

blend_score = roc_auc_score(
    y,
    blend_333
)

print(
    f"Equal Blend OOF ROC-AUC: "
    f"{blend_score:.6f}"
)

Equal Blend OOF ROC-AUC: 0.964577


In [28]:
weights = [
    (0.60, 0.20, 0.20),
    (0.50, 0.25, 0.25),
    (0.50, 0.20, 0.30),
    (0.40, 0.30, 0.30),
    (0.40, 0.20, 0.40),
    (0.30, 0.30, 0.40),
    (0.30, 0.40, 0.30),
    (0.20, 0.40, 0.40),
    (0.25, 0.25, 0.50),
]

blend_results = []

for lw, xw, cw in weights:

    prediction = (
        lw * lgb_oof +
        xw * xgb_oof +
        cw * cat_oof
    )

    score = roc_auc_score(
        y,
        prediction
    )

    blend_results.append({
        "LightGBM": lw,
        "XGBoost": xw,
        "CatBoost": cw,
        "ROC-AUC": score
    })

blend_results = pd.DataFrame(
    blend_results
)

blend_results.sort_values(
    "ROC-AUC",
    ascending=False
)

,LightGBM,XGBoost,CatBoost,ROC-AUC
6,0.30,0.40,0.30,0.964594
7,0.20,0.40,0.40,0.964584
5,0.30,0.30,0.40,0.964556
3,0.40,0.30,0.30,0.964553
1,0.50,0.25,0.25,0.964490
8,0.25,0.25,0.50,0.964485
4,0.40,0.20,0.40,0.964468
2,0.50,0.20,0.30,0.964453
0,0.60,0.20,0.20,0.964395


In [29]:
best_blend = blend_results.sort_values(
    "ROC-AUC",
    ascending=False
).iloc[0]

print(best_blend)

LightGBM    0.300000
XGBoost     0.400000
CatBoost    0.300000
ROC-AUC     0.964594
Name: 6, dtype: float64


In [30]:
best_lgb_weight = best_blend["LightGBM"]
best_xgb_weight = best_blend["XGBoost"]
best_cat_weight = best_blend["CatBoost"]

In [31]:
final_test_prediction = (
    best_lgb_weight * lgb_test +
    best_xgb_weight * xgb_test +
    best_cat_weight * cat_test
)

In [32]:
print(
    "Minimum:",
    final_test_prediction.min()
)

print(
    "Maximum:",
    final_test_prediction.max()
)

print(
    "Mean:",
    final_test_prediction.mean()
)

Minimum: 5.8180295077738626e-05
Maximum: 0.9999998505270016
Mean: 0.7100172850630503


In [33]:
print(X.columns.tolist())

['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact']


In [34]:
print(X.dtypes)

age                         float64
daily_screen_time_hours     float64
social_media_hours          float64
gaming_hours                float64
work_study_hours            float64
sleep_hours                 float64
notifications_per_day       float64
app_opens_per_day           float64
weekend_screen_time         float64
gender                     category
stress_level               category
academic_work_impact       category
dtype: object


In [35]:
print(train.head())

   id   age  daily_screen_time_hours  social_media_hours  gaming_hours  \
0   0  24.0                      NaN                1.83          1.59   
1   1  19.0                     5.97                1.08           NaN   
2   2  18.0                     5.09                 NaN           NaN   
3   3  21.0                     6.42                1.26          1.42   
4   4  26.0                    11.20                1.87          2.81   

   work_study_hours  sleep_hours  notifications_per_day  app_opens_per_day  \
0              2.11         7.46                  122.0               38.0   
1              3.03         8.22                   76.0               19.0   
2               NaN         6.25                  134.0               60.0   
3              3.36         8.85                  112.0               94.0   
4              1.95         5.25                    NaN                NaN   

   weekend_screen_time  gender stress_level academic_work_impact  \
0                 

In [36]:
numeric_columns = X.select_dtypes(
    include=["number"]
).columns

for col in numeric_columns:
    score = roc_auc_score(
        y,
        X[col].fillna(X[col].median())
    )

    score = max(score, 1 - score)

    print(
        f"{col:35s} {score:.6f}"
    )

age                                 0.502115
daily_screen_time_hours             0.865372
social_media_hours                  0.817831
gaming_hours                        0.608820
work_study_hours                    0.648551
sleep_hours                         0.525288
notifications_per_day               0.507757
app_opens_per_day                   0.537697
weekend_screen_time                 0.850235


In [37]:
for col in categorical_columns:

    temp = (
        train.groupby(col)["addicted_label"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
    )

    print("\n" + "=" * 60)
    print(col)
    print("=" * 60)
    print(temp.head(15))


gender
            mean   count
gender                  
Male    0.723198  223662
Female  0.703838  221595
Other   0.701020  217078

stress_level
                  mean   count
stress_level                  
High          0.711431  220873
Low           0.711160  207783
Medium        0.705663  207565

academic_work_impact
                          mean   count
academic_work_impact                  
No                    0.711029  316579
Yes                   0.707883  330566


In [38]:
submission = sample_submission.copy()

submission["addicted_label"] = final_test_prediction

submission.head()

,id,addicted_label
0,691369,0.999540
1,691370,0.947453
2,691371,0.947776
3,691372,0.991855
4,691373,0.997833


In [39]:
print(
    "Shape:",
    submission.shape
)

print(
    "Columns:",
    submission.columns.tolist()
)

print(
    "Missing:",
    submission["addicted_label"].isna().sum()
)

print(
    "Duplicate IDs:",
    submission["id"].duplicated().sum()
)

Shape: (296302, 2)
Columns: ['id', 'addicted_label']
Missing: 0
Duplicate IDs: 0


In [40]:
submission.to_csv(
    "smartphone_addiction_ensemble_improved.csv",
    index=False
)

print(
    "Saved successfully!"
)

Saved successfully!


In [41]:
X_fe = X.copy()
X_test_fe = X_test.copy()

In [42]:
for df in [X_fe, X_test_fe]:

    df["weekend_screen_diff"] = (
        df["weekend_screen_time"]
        - df["daily_screen_time_hours"]
    )

    df["weekend_screen_ratio"] = (
        df["weekend_screen_time"]
        / (df["daily_screen_time_hours"] + 0.1)
    )

    df["recreational_hours"] = (
        df["social_media_hours"]
        + df["gaming_hours"]
    )

    df["recreational_ratio"] = (
        df["recreational_hours"]
        / (df["daily_screen_time_hours"] + 0.1)
    )

    df["non_recreational_screen"] = (
        df["daily_screen_time_hours"]
        - df["recreational_hours"]
    )

In [43]:
for df in [X_fe, X_test_fe]:

    df["social_gaming_ratio"] = (
        df["social_media_hours"]
        / (df["gaming_hours"] + 0.1)
    )

    df["social_screen_ratio"] = (
        df["social_media_hours"]
        / (df["daily_screen_time_hours"] + 0.1)
    )

    df["gaming_screen_ratio"] = (
        df["gaming_hours"]
        / (df["daily_screen_time_hours"] + 0.1)
    )

In [44]:
for df in [X_fe, X_test_fe]:

    df["screen_sleep_ratio"] = (
        df["daily_screen_time_hours"]
        / (df["sleep_hours"] + 0.1)
    )

    df["screen_sleep_difference"] = (
        df["daily_screen_time_hours"]
        - df["sleep_hours"]
    )

    df["sleep_deficit_proxy"] = (
        8.0 - df["sleep_hours"]
    )

In [45]:
for df in [X_fe, X_test_fe]:

    df["notifications_per_hour"] = (
        df["notifications_per_day"]
        / (df["daily_screen_time_hours"] + 0.1)
    )

    df["opens_per_hour"] = (
        df["app_opens_per_day"]
        / (df["daily_screen_time_hours"] + 0.1)
    )

    df["notifications_per_open"] = (
        df["notifications_per_day"]
        / (df["app_opens_per_day"] + 0.1)
    )

In [46]:
for df in [X_fe, X_test_fe]:

    df["total_activity_hours"] = (
        df["social_media_hours"]
        + df["gaming_hours"]
        + df["work_study_hours"]
    )

    df["screen_plus_activity"] = (
        df["daily_screen_time_hours"]
        + df["social_media_hours"]
        + df["gaming_hours"]
    )

    df["screen_to_activity_ratio"] = (
        df["daily_screen_time_hours"]
        / (df["total_activity_hours"] + 0.1)
    )

In [47]:
for df in [X_fe, X_test_fe]:

    df["screen_time_sq"] = (
        df["daily_screen_time_hours"] ** 2
    )

    df["screen_time_cu"] = (
        df["daily_screen_time_hours"] ** 3
    )

    df["weekend_screen_sq"] = (
        df["weekend_screen_time"] ** 2
    )

    df["social_media_sq"] = (
        df["social_media_hours"] ** 2
    )

    df["gaming_sq"] = (
        df["gaming_hours"] ** 2
    )

In [48]:
print("Original features:", X.shape[1])
print("Engineered features:", X_fe.shape[1])
print("\nNew features:")

new_features = [
    col for col in X_fe.columns
    if col not in X.columns
]

print(new_features)

Original features: 12
Engineered features: 34

New features:
['weekend_screen_diff', 'weekend_screen_ratio', 'recreational_hours', 'recreational_ratio', 'non_recreational_screen', 'social_gaming_ratio', 'social_screen_ratio', 'gaming_screen_ratio', 'screen_sleep_ratio', 'screen_sleep_difference', 'sleep_deficit_proxy', 'notifications_per_hour', 'opens_per_hour', 'notifications_per_open', 'total_activity_hours', 'screen_plus_activity', 'screen_to_activity_ratio', 'screen_time_sq', 'screen_time_cu', 'weekend_screen_sq', 'social_media_sq', 'gaming_sq']


In [49]:
lgb_fe_oof = np.zeros(len(X_fe))
lgb_fe_test = np.zeros(len(X_test_fe))

In [50]:
for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print("=" * 60)
    print(f"FOLD {fold}/5")
    print("=" * 60)

    X_train_fe = X_fe.iloc[train_idx]
    X_valid_fe = X_fe.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMClassifier(
        **lgb_params
    )

    model.fit(
        X_train_fe,
        y_train,

        eval_set=[
            (X_valid_fe, y_valid)
        ],

        eval_metric="auc",

        categorical_feature=categorical_columns,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=200,
                verbose=True
            ),
            lgb.log_evaluation(
                period=100
            )
        ]
    )

    valid_pred = model.predict_proba(
        X_valid_fe
    )[:, 1]

    test_pred = model.predict_proba(
        X_test_fe
    )[:, 1]

    lgb_fe_oof[valid_idx] = valid_pred

    lgb_fe_test += (
        test_pred / len(folds)
    )

    fold_auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    print(
        f"Fold {fold} ROC-AUC: {fold_auc:.6f}"
    )

FOLD 1/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.944716
[200]	valid_0's auc: 0.955218
[300]	valid_0's auc: 0.958966
[400]	valid_0's auc: 0.960575
[500]	valid_0's auc: 0.961239
[600]	valid_0's auc: 0.961524
[700]	valid_0's auc: 0.961797
[800]	valid_0's auc: 0.962134
[900]	valid_0's auc: 0.962268
[1000]	valid_0's auc: 0.962373
[1100]	valid_0's auc: 0.962501
[1200]	valid_0's auc: 0.962632
[1300]	valid_0's auc: 0.96274
[1400]	valid_0's auc: 0.962836
[1500]	valid_0's auc: 0.96289
[1600]	valid_0's auc: 0.96294
[1700]	valid_0's auc: 0.962979
[1800]	valid_0's auc: 0.963011
[1900]	valid_0's auc: 0.963054
[2000]	valid_0's auc: 0.963073
[2100]	valid_0's auc: 0.963087
[2200]	valid_0's auc: 0.963128
[2300]	valid_0's auc: 0.963149
[2400]	valid_0's auc: 0.963169
[2500]	valid_0's auc: 0.963181
[2600]	valid_0's auc: 0.963181
[2700]	valid_0's auc: 0.963184
[2800]	valid_0's auc: 0.963185
[2900]	valid_0's auc: 0.963181
Early stopping, best iteration is:
[2795]	valid_0's

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.945114
[200]	valid_0's auc: 0.955862
[300]	valid_0's auc: 0.959692
[400]	valid_0's auc: 0.961345
[500]	valid_0's auc: 0.961995
[600]	valid_0's auc: 0.962277
[700]	valid_0's auc: 0.962561
[800]	valid_0's auc: 0.962742
[900]	valid_0's auc: 0.963023
[1000]	valid_0's auc: 0.963167
[1100]	valid_0's auc: 0.963305
[1200]	valid_0's auc: 0.963379
[1300]	valid_0's auc: 0.963465
[1400]	valid_0's auc: 0.963548
[1500]	valid_0's auc: 0.963634
[1600]	valid_0's auc: 0.963692
[1700]	valid_0's auc: 0.963715
[1800]	valid_0's auc: 0.963759
[1900]	valid_0's auc: 0.96381
[2000]	valid_0's auc: 0.96385
[2100]	valid_0's auc: 0.963889
[2200]	valid_0's auc: 0.963919
[2300]	valid_0's auc: 0.963939
[2400]	valid_0's auc: 0.963955
[2500]	valid_0's auc: 0.963983
[2600]	valid_0's auc: 0.963977
[2700]	valid_0's auc: 0.963969
Early stopping, best iteration is:
[2520]	valid_0's auc: 0.963984
Fold 2 ROC-AUC: 0.963984
FOLD 3/5


...

Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.945663
[200]	valid_0's auc: 0.95637
[300]	valid_0's auc: 0.960114
[400]	valid_0's auc: 0.961782
[500]	valid_0's auc: 0.962445
[600]	valid_0's auc: 0.962761
[700]	valid_0's auc: 0.963007
[800]	valid_0's auc: 0.963267
[900]	valid_0's auc: 0.963428
[1000]	valid_0's auc: 0.96356
[1100]	valid_0's auc: 0.963687
[1200]	valid_0's auc: 0.963771
[1300]	valid_0's auc: 0.963857
[1400]	valid_0's auc: 0.963912
[1500]	valid_0's auc: 0.963971
[1600]	valid_0's auc: 0.964056
[1700]	valid_0's auc: 0.964106
[1800]	valid_0's auc: 0.964147
[1900]	valid_0's auc: 0.964157
[2000]	valid_0's auc: 0.964171
[2100]	valid_0's auc: 0.964193
[2200]	valid_0's auc: 0.964216
[2300]	valid_0's auc: 0.964235
[2400]	valid_0's auc: 0.96428
[2500]	valid_0's auc: 0.964295
[2600]	valid_0's auc: 0.964334
[2700]	valid_0's auc: 0.964329
[2800]	valid_0's auc: 0.964336
[2900]	valid_0's auc: 0.964331
Early stopping, best iteration is:
[2775]	valid_0's

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.946392
[200]	valid_0's auc: 0.957239
[300]	valid_0's auc: 0.960924
[400]	valid_0's auc: 0.962387
[500]	valid_0's auc: 0.963112
[600]	valid_0's auc: 0.963457
[700]	valid_0's auc: 0.963724
[800]	valid_0's auc: 0.963978
[900]	valid_0's auc: 0.964094
[1000]	valid_0's auc: 0.964203
[1100]	valid_0's auc: 0.964355
[1200]	valid_0's auc: 0.964459
[1300]	valid_0's auc: 0.964542
[1400]	valid_0's auc: 0.964625
[1500]	valid_0's auc: 0.964674
[1600]	valid_0's auc: 0.964725
[1700]	valid_0's auc: 0.964801
[1800]	valid_0's auc: 0.964857
[1900]	valid_0's auc: 0.964879
[2000]	valid_0's auc: 0.964899
[2100]	valid_0's auc: 0.964942
[2200]	valid_0's auc: 0.964963
[2300]	valid_0's auc: 0.964989
[2400]	valid_0's auc: 0.964993
[2500]	valid_0's auc: 0.965003
[2600]	valid_0's auc: 0.964999
[2700]	valid_0's auc: 0.965005
[2800]	valid_0's auc: 0.964995
Early stopping, best iteration is:
[2675]	valid_0's auc: 0.965008
Fold 4 ROC-AU

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.94558
[200]	valid_0's auc: 0.956034
[300]	valid_0's auc: 0.959686
[400]	valid_0's auc: 0.961265
[500]	valid_0's auc: 0.961958
[600]	valid_0's auc: 0.962326
[700]	valid_0's auc: 0.962697
[800]	valid_0's auc: 0.962923
[900]	valid_0's auc: 0.963132
[1000]	valid_0's auc: 0.963222
[1100]	valid_0's auc: 0.963386
[1200]	valid_0's auc: 0.963503
[1300]	valid_0's auc: 0.963576
[1400]	valid_0's auc: 0.963653
[1500]	valid_0's auc: 0.963714
[1600]	valid_0's auc: 0.96377
[1700]	valid_0's auc: 0.963809
[1800]	valid_0's auc: 0.963849
[1900]	valid_0's auc: 0.963854
[2000]	valid_0's auc: 0.963879
[2100]	valid_0's auc: 0.963902
[2200]	valid_0's auc: 0.963908
[2300]	valid_0's auc: 0.963915
[2400]	valid_0's auc: 0.963901
Early stopping, best iteration is:
[2253]	valid_0's auc: 0.963918
Fold 5 ROC-AUC: 0.963918


In [51]:
lgb_fe_auc = roc_auc_score(
    y,
    lgb_fe_oof
)

print(
    f"Feature Engineered LightGBM OOF ROC-AUC: "
    f"{lgb_fe_auc:.6f}"
)

Feature Engineered LightGBM OOF ROC-AUC: 0.964084


In [52]:
feature_importance = pd.DataFrame({
    "feature": X_fe.columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(feature_importance.to_string(index=False))

                 feature  importance
   notifications_per_day       13837
       app_opens_per_day       12030
     weekend_screen_time        9576
        work_study_hours        8209
 daily_screen_time_hours        6229
      social_media_hours        5811
             sleep_hours        5498
screen_to_activity_ratio        5337
  notifications_per_open        4856
     social_screen_ratio        4678
                     age        4592
            gaming_hours        4503
     gaming_screen_ratio        4367
  notifications_per_hour        4274
          opens_per_hour        4104
     social_gaming_ratio        4023
     weekend_screen_diff        3629
    total_activity_hours        3425
    weekend_screen_ratio        3314
     sleep_deficit_proxy        3312
      recreational_ratio        3272
 screen_sleep_difference        3083
      screen_sleep_ratio        3060
      recreational_hours        2629
 non_recreational_screen        2534
    screen_plus_activity        2295
 

In [53]:
original_features = set(X.columns)

feature_importance["type"] = feature_importance["feature"].apply(
    lambda x: "Original" if x in original_features else "Engineered"
)

print(
    feature_importance.groupby("type")["importance"]
    .sum()
)

type
Engineered    66313
Original      73373
Name: importance, dtype: int32


In [54]:
interaction_features = [
    ("notifications_per_day", "daily_screen_time_hours"),
    ("app_opens_per_day", "daily_screen_time_hours"),
    ("notifications_per_day", "app_opens_per_day"),
    ("social_media_hours", "daily_screen_time_hours"),
    ("gaming_hours", "daily_screen_time_hours"),
    ("work_study_hours", "daily_screen_time_hours"),
    ("sleep_hours", "daily_screen_time_hours"),
    ("social_media_hours", "weekend_screen_time"),
    ("gaming_hours", "weekend_screen_time"),
    ("notifications_per_day", "social_media_hours"),
    ("app_opens_per_day", "social_media_hours"),
]

In [55]:
for df in [X_fe, X_test_fe]:

    for col1, col2 in interaction_features:

        new_col = f"{col1}_x_{col2}"

        df[new_col] = (
            df[col1] * df[col2]
        )

In [56]:
for df in [X_fe, X_test_fe]:

    df["social_share_of_screen"] = (
        df["social_media_hours"]
        / df["daily_screen_time_hours"]
    )

    df["gaming_share_of_screen"] = (
        df["gaming_hours"]
        / df["daily_screen_time_hours"]
    )

    df["social_plus_gaming_share"] = (
        (
            df["social_media_hours"]
            + df["gaming_hours"]
        )
        / df["daily_screen_time_hours"]
    )

In [57]:
print(
    "X_fe infinite values:",
    np.isinf(X_fe.select_dtypes(include=np.number)).sum().sum()
)

print(
    "X_test_fe infinite values:",
    np.isinf(X_test_fe.select_dtypes(include=np.number)).sum().sum()
)

X_fe infinite values: 0
X_test_fe infinite values: 0


In [58]:
print("Original features:", X.shape[1])
print("Feature-engineered features:", X_fe.shape[1])
print("Test features:", X_test_fe.shape[1])

Original features: 12
Feature-engineered features: 48
Test features: 48


In [59]:
lgb_strong_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",

    "n_estimators": 5000,
    "learning_rate": 0.03,

    "num_leaves": 127,
    "max_depth": -1,

    "min_child_samples": 20,

    "subsample": 0.85,
    "colsample_bytree": 0.85,

    "reg_alpha": 0.1,
    "reg_lambda": 1.0,

    "random_state": 42,

    "n_jobs": -1,
    "verbosity": -1
}

In [60]:
lgb_strong_oof = np.zeros(len(X_fe))
lgb_strong_test = np.zeros(len(X_test_fe))

In [61]:
for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print("=" * 70)
    print(f"FOLD {fold}/5")
    print("=" * 70)

    X_train = X_fe.iloc[train_idx]
    X_valid = X_fe.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMClassifier(
        **lgb_strong_params
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (X_valid, y_valid)
        ],

        eval_metric="auc",

        categorical_feature=categorical_columns,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=200,
                verbose=True
            ),
            lgb.log_evaluation(
                period=100
            )
        ]
    )

    valid_pred = model.predict_proba(
        X_valid
    )[:, 1]

    test_pred = model.predict_proba(
        X_test_fe
    )[:, 1]

    lgb_strong_oof[valid_idx] = valid_pred

    lgb_strong_test += (
        test_pred / len(folds)
    )

    fold_auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    print(
        f"Fold {fold} ROC-AUC: {fold_auc:.6f}"
    )

FOLD 1/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.948205
[200]	valid_0's auc: 0.958722
[300]	valid_0's auc: 0.961145
[400]	valid_0's auc: 0.962191
[500]	valid_0's auc: 0.962428
[600]	valid_0's auc: 0.962599
[700]	valid_0's auc: 0.962708
[800]	valid_0's auc: 0.962807
[900]	valid_0's auc: 0.962876
[1000]	valid_0's auc: 0.962906
[1100]	valid_0's auc: 0.962973
[1200]	valid_0's auc: 0.963027
[1300]	valid_0's auc: 0.963015
[1400]	valid_0's auc: 0.96302
Early stopping, best iteration is:
[1201]	valid_0's auc: 0.963028
Fold 1 ROC-AUC: 0.963028
FOLD 2/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.948483
[200]	valid_0's auc: 0.959412
[300]	valid_0's auc: 0.961832
[400]	valid_0's auc: 0.962773
[500]	valid_0's auc: 0.963096
[600]	valid_0's auc: 0.963187
[700]	valid_0's auc: 0.96335
[800]	valid_0's auc: 0.963465
[900]	valid_0's auc: 0.963535
[1000]	valid_0's auc: 0.963563
[1100]	valid_0's auc: 0.963612
[1200]	valid_0's auc: 0.963626
[1300]	valid_0's auc: 0.963647
[1400]	valid_0's auc: 0.963665
[1500]	valid_0's auc: 0.963668
Early stopping, best iteration is:
[1353]	valid_0's auc: 0.963671
Fold 2 ROC-AUC: 0.963671
FOLD 3/5


...

Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.949774
[200]	valid_0's auc: 0.959651
[300]	valid_0's auc: 0.962259
[400]	valid_0's auc: 0.96305
[500]	valid_0's auc: 0.963376
[600]	valid_0's auc: 0.963555
[700]	valid_0's auc: 0.963649
[800]	valid_0's auc: 0.963774
[900]	valid_0's auc: 0.963806
[1000]	valid_0's auc: 0.963866
[1100]	valid_0's auc: 0.96391
[1200]	valid_0's auc: 0.963926
[1300]	valid_0's auc: 0.963938
[1400]	valid_0's auc: 0.963935
[1500]	valid_0's auc: 0.963938
[1600]	valid_0's auc: 0.963934
[1700]	valid_0's auc: 0.963937
[1800]	valid_0's auc: 0.963908
Early stopping, best iteration is:
[1616]	valid_0's auc: 0.963949
Fold 3 ROC-AUC: 0.963949
FOLD 4/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.950112
[200]	valid_0's auc: 0.960374
[300]	valid_0's auc: 0.962957
[400]	valid_0's auc: 0.963892
[500]	valid_0's auc: 0.964127
[600]	valid_0's auc: 0.964248
[700]	valid_0's auc: 0.964345
[800]	valid_0's auc: 0.964421
[900]	valid_0's auc: 0.964457
[1000]	valid_0's auc: 0.964518
[1100]	valid_0's auc: 0.964574
[1200]	valid_0's auc: 0.964566
[1300]	valid_0's auc: 0.964583
[1400]	valid_0's auc: 0.964585
[1500]	valid_0's auc: 0.964599
[1600]	valid_0's auc: 0.964602
[1700]	valid_0's auc: 0.9646
Early stopping, best iteration is:
[1536]	valid_0's auc: 0.964613
Fold 4 ROC-AUC: 0.964613
FOLD 5/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.949275
[200]	valid_0's auc: 0.959204
[300]	valid_0's auc: 0.961775
[400]	valid_0's auc: 0.962636
[500]	valid_0's auc: 0.962908
[600]	valid_0's auc: 0.963105
[700]	valid_0's auc: 0.963267
[800]	valid_0's auc: 0.963378
[900]	valid_0's auc: 0.963448
[1000]	valid_0's auc: 0.963533
[1100]	valid_0's auc: 0.963573
[1200]	valid_0's auc: 0.963596
[1300]	valid_0's auc: 0.963628
[1400]	valid_0's auc: 0.963674
[1500]	valid_0's auc: 0.963714
[1600]	valid_0's auc: 0.963716
[1700]	valid_0's auc: 0.963715
[1800]	valid_0's auc: 0.9637
Early stopping, best iteration is:
[1623]	valid_0's auc: 0.963724
Fold 5 ROC-AUC: 0.963724


In [62]:
lgb_strong_auc = roc_auc_score(
    y,
    lgb_strong_oof
)

print(
    f"Strong LightGBM OOF ROC-AUC: "
    f"{lgb_strong_auc:.6f}"
)

Strong LightGBM OOF ROC-AUC: 0.963793


In [63]:
results = pd.DataFrame({
    "Model": [
        "Original LightGBM",
        "Feature Engineered LightGBM",
        "Previous Ensemble",
        "Strong Feature Engineered LightGBM"
    ],

    "OOF ROC-AUC": [
        0.963683,
        0.964084,
        0.964594,
        lgb_strong_auc
    ]
})

print(
    results.sort_values(
        "OOF ROC-AUC",
        ascending=False
    ).to_string(index=False)
)

                             Model  OOF ROC-AUC
                 Previous Ensemble     0.964594
       Feature Engineered LightGBM     0.964084
Strong Feature Engineered LightGBM     0.963793
                 Original LightGBM     0.963683


In [64]:
X_num = X.copy()
X_test_num = X_test.copy()

for col in categorical_columns:
    X_num[col] = (
        X_num[col]
        .astype(str)
        .fillna("MISSING")
    )

    X_test_num[col] = (
        X_test_num[col]
        .astype(str)
        .fillna("MISSING")
    )

X_num = pd.get_dummies(
    X_num,
    columns=categorical_columns,
    dummy_na=True
)

X_test_num = pd.get_dummies(
    X_test_num,
    columns=categorical_columns,
    dummy_na=True
)

X_test_num = X_test_num.reindex(
    columns=X_num.columns,
    fill_value=0
)

In [65]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [66]:
lr_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            C=1.0,
            max_iter=1000,
            n_jobs=-1
        )
    )
])

In [67]:
lr_oof = np.zeros(len(X_num))

for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print(f"Fold {fold}/5")

    lr_pipeline.fit(
        X_num.iloc[train_idx],
        y.iloc[train_idx]
    )

    valid_pred = lr_pipeline.predict_proba(
        X_num.iloc[valid_idx]
    )[:, 1]

    lr_oof[valid_idx] = valid_pred

    print(
        f"Fold AUC: "
        f"{roc_auc_score(y.iloc[valid_idx], valid_pred):.6f}"
    )

Fold 1/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Fold AUC: 0.910389
Fold 2/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Fold AUC: 0.910816
Fold 3/5


...

Fold AUC: 0.911926
Fold 4/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Fold AUC: 0.912668
Fold 5/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Fold AUC: 0.911603


In [68]:
lr_auc = roc_auc_score(
    y,
    lr_oof
)

print(
    f"Logistic Regression OOF ROC-AUC: "
    f"{lr_auc:.6f}"
)

Logistic Regression OOF ROC-AUC: 0.911478


In [69]:
X_missing = X.copy()
X_test_missing = X_test.copy()

for col in X.columns:

    X_missing[f"{col}_missing"] = (
        X[col].isna().astype("int8")
    )

    X_test_missing[f"{col}_missing"] = (
        X_test[col].isna().astype("int8")
    )

In [70]:
print("Original features:", X.shape[1])
print("With missing indicators:", X_missing.shape[1])

Original features: 12
With missing indicators: 24


In [71]:
missing_auc = {}

for col in X.columns:

    missing_indicator = X[col].isna().astype(int)

    auc = roc_auc_score(
        y,
        missing_indicator
    )

    auc = max(auc, 1 - auc)

    missing_auc[col] = auc

missing_auc = (
    pd.Series(missing_auc)
    .sort_values(ascending=False)
)

print(missing_auc)

app_opens_per_day          0.500952
daily_screen_time_hours    0.500631
sleep_hours                0.500617
weekend_screen_time        0.500483
gaming_hours               0.500481
age                        0.500404
work_study_hours           0.500221
notifications_per_day      0.500194
stress_level               0.500082
gender                     0.500065
academic_work_impact       0.500005
social_media_hours         0.500005
dtype: float64


In [72]:
for col in X.columns:

    missing_rate = y[X[col].isna()].mean()
    present_rate = y[X[col].notna()].mean()

    print(
        f"{col:35s} "
        f"Missing: {missing_rate:.6f} | "
        f"Present: {present_rate:.6f} | "
        f"Difference: {missing_rate - present_rate:+.6f}"
    )

age                                 Missing: 0.713402 | Present: 0.709251 | Difference: +0.004151
daily_screen_time_hours             Missing: 0.711301 | Present: 0.709122 | Difference: +0.002178
social_media_hours                  Missing: 0.709415 | Present: 0.709427 | Difference: -0.000012
gaming_hours                        Missing: 0.710505 | Present: 0.709182 | Difference: +0.001324
work_study_hours                    Missing: 0.710645 | Present: 0.709326 | Difference: +0.001319
sleep_hours                         Missing: 0.713377 | Present: 0.709153 | Difference: +0.004224
notifications_per_day               Missing: 0.710242 | Present: 0.709336 | Difference: +0.000906
app_opens_per_day                   Missing: 0.712787 | Present: 0.708980 | Difference: +0.003807
weekend_screen_time                 Missing: 0.710654 | Present: 0.709187 | Difference: +0.001467
gender                              Missing: 0.708790 | Present: 0.709452 | Difference: -0.000662
stress_level        

In [73]:
temp = train.copy()

temp["screen_bin"] = pd.qcut(
    temp["daily_screen_time_hours"],
    q=20,
    duplicates="drop"
)

screen_stats = (
    temp.groupby(
        "screen_bin",
        observed=True
    )["addicted_label"]
    .agg(
        ["mean", "count"]
    )
)

print(screen_stats.to_string())

                    mean  count
screen_bin                     
(0.499, 3.19]   0.232704  29905
(3.19, 3.97]    0.297188  30119
(3.97, 4.46]    0.290117  29657
(4.46, 4.96]    0.330658  29650
(4.96, 5.48]    0.377287  29903
(5.48, 5.89]    0.420108  29540
(5.89, 6.32]    0.428801  29860
(6.32, 6.8]     0.640332  29733
(6.8, 7.27]     0.696504  29638
(7.27, 7.77]    0.749975  30033
(7.77, 8.26]    0.859709  30230
(8.26, 8.7]     0.931898  29441
(8.7, 9.12]     0.954263  29495
(9.12, 9.47]    0.993509  30657
(9.47, 9.84]    0.993439  28960
(9.84, 10.25]   0.995994  29957
(10.25, 10.74]  0.998087  29799
(10.74, 11.24]  0.999232  29931
(11.24, 11.71]  0.999593  29491
(11.71, 15.0]   0.999729  29516


In [74]:
temp["weekend_bin"] = pd.qcut(
    temp["weekend_screen_time"],
    q=20,
    duplicates="drop"
)

weekend_stats = (
    temp.groupby(
        "weekend_bin",
        observed=True
    )["addicted_label"]
    .agg(
        ["mean", "count"]
    )
)

print(weekend_stats.to_string())

                    mean  count
weekend_bin                    
(0.509, 4.72]   0.262568  29021
(4.72, 5.63]    0.266273  28989
(5.63, 6.17]    0.297605  29183
(6.17, 6.76]    0.346699  29080
(6.76, 7.28]    0.390026  28575
(7.28, 7.73]    0.401960  30605
(7.73, 8.2]     0.514759  27407
(8.2, 8.68]     0.626443  29634
(8.68, 9.12]    0.702546  30203
(9.12, 9.58]    0.794123  27769
(9.58, 10.01]   0.824942  28659
(10.01, 10.47]  0.909243  29871
(10.47, 10.86]  0.929162  27810
(10.86, 11.33]  0.966839  29432
(11.33, 11.75]  0.982109  28897
(11.75, 12.08]  0.993989  28946
(12.08, 12.64]  0.996746  29505
(12.64, 13.19]  0.999247  27894
(13.19, 13.96]  0.999759  29036
(13.96, 17.56]  0.999618  28790


In [75]:
temp["social_bin"] = pd.qcut(
    temp["social_media_hours"],
    q=20,
    duplicates="drop"
)

social_stats = (
    temp.groupby(
        "social_bin",
        observed=True
    )["addicted_label"]
    .agg(
        ["mean", "count"]
    )
)

print(social_stats.to_string())

                    mean  count
social_bin                     
(-0.001, 0.58]  0.214604  28513
(0.58, 0.85]    0.285565  27780
(0.85, 1.07]    0.310569  27939
(1.07, 1.27]    0.364721  28334
(1.27, 1.45]    0.441813  27867
(1.45, 1.62]    0.518333  26919
(1.62, 1.79]    0.587051  28621
(1.79, 1.94]    0.678996  27224
(1.94, 2.12]    0.711479  27686
(2.12, 2.31]    0.745685  28044
(2.31, 2.51]    0.806794  29233
(2.51, 2.7]     0.857242  27284
(2.7, 2.9]      0.889374  27941
(2.9, 3.16]     0.910255  27946
(3.16, 3.37]    0.954510  26907
(3.37, 3.65]    0.964565  28559
(3.65, 3.94]    0.980737  27046
(3.94, 4.33]    0.990018  28251
(4.33, 4.86]    0.997603  27536
(4.86, 8.0]     0.999676  27744


In [76]:
strong_features = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "social_media_hours",
    "work_study_hours",
    "gaming_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "age"
]

X_hgb = X[strong_features].copy()
X_test_hgb = X_test[strong_features].copy()

print(X_hgb.shape)

(691369, 9)


In [77]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_params = {
    "learning_rate": 0.05,
    "max_iter": 1000,
    "max_leaf_nodes": 63,
    "max_depth": None,
    "min_samples_leaf": 30,
    "l2_regularization": 1.0,
    "random_state": 42
}

In [78]:
hgb_oof = np.zeros(len(X_hgb))
hgb_test = np.zeros(len(X_test_hgb))

In [79]:
for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print("=" * 60)
    print(f"FOLD {fold}/5")
    print("=" * 60)

    model_hgb = HistGradientBoostingClassifier(
        **hgb_params
    )

    model_hgb.fit(
        X_hgb.iloc[train_idx],
        y.iloc[train_idx]
    )

    valid_pred = model_hgb.predict_proba(
        X_hgb.iloc[valid_idx]
    )[:, 1]

    test_pred = model_hgb.predict_proba(
        X_test_hgb
    )[:, 1]

    hgb_oof[valid_idx] = valid_pred

    hgb_test += (
        test_pred / len(folds)
    )

    fold_auc = roc_auc_score(
        y.iloc[valid_idx],
        valid_pred
    )

    print(
        f"Fold {fold} ROC-AUC: {fold_auc:.6f}"
    )

FOLD 1/5
Fold 1 ROC-AUC: 0.961583
FOLD 2/5
Fold 2 ROC-AUC: 0.962739
FOLD 3/5
Fold 3 ROC-AUC: 0.962585
FOLD 4/5
Fold 4 ROC-AUC: 0.963511
FOLD 5/5
Fold 5 ROC-AUC: 0.962187


In [80]:
hgb_auc = roc_auc_score(
    y,
    hgb_oof
)

print(
    f"HistGradientBoosting OOF ROC-AUC: "
    f"{hgb_auc:.6f}"
)

HistGradientBoosting OOF ROC-AUC: 0.962519


In [81]:
thresholds = {
    "daily_screen_time_hours": [6.3, 7.0, 7.8, 8.3, 9.1, 10.0],
    "weekend_screen_time": [7.7, 8.2, 8.7, 9.1, 10.0, 11.0],
    "social_media_hours": [1.4, 1.6, 1.8, 2.1, 2.5, 3.0]
}

for df in [X_fe, X_test_fe]:

    for col, values in thresholds.items():

        for threshold in values:

            safe_name = str(threshold).replace(".", "_")

            df[f"{col}_gt_{safe_name}"] = (
                df[col] > threshold
            ).astype("int8")

In [82]:
for df in [X_fe, X_test_fe]:

    for col, values in thresholds.items():

        for threshold in values:

            safe_name = str(threshold).replace(".", "_")

            df[f"{col}_dist_{safe_name}"] = (
                df[col] - threshold
            )

In [83]:
print("Training features:", X_fe.shape[1])
print("Test features:", X_test_fe.shape[1])

Training features: 84
Test features: 84


In [84]:
threshold_oof = np.zeros(len(X_fe))
threshold_test = np.zeros(len(X_test_fe))

In [85]:
threshold_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",

    "n_estimators": 5000,
    "learning_rate": 0.03,

    "num_leaves": 63,
    "max_depth": -1,

    "min_child_samples": 30,

    "subsample": 0.85,
    "colsample_bytree": 0.85,

    "reg_alpha": 0.1,
    "reg_lambda": 1.0,

    "random_state": 42,

    "n_jobs": -1,
    "verbosity": -1
}

In [86]:
for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print("=" * 60)
    print(f"FOLD {fold}/5")
    print("=" * 60)

    X_train = X_fe.iloc[train_idx]
    X_valid = X_fe.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMClassifier(
        **threshold_params
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (X_valid, y_valid)
        ],

        eval_metric="auc",

        categorical_feature=categorical_columns,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=200,
                verbose=True
            ),
            lgb.log_evaluation(
                period=100
            )
        ]
    )

    valid_pred = model.predict_proba(
        X_valid
    )[:, 1]

    test_pred = model.predict_proba(
        X_test_fe
    )[:, 1]

    threshold_oof[valid_idx] = valid_pred

    threshold_test += (
        test_pred / len(folds)
    )

    fold_auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    print(
        f"Fold {fold} ROC-AUC: {fold_auc:.6f}"
    )

FOLD 1/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.945048
[200]	valid_0's auc: 0.955537
[300]	valid_0's auc: 0.95949
[400]	valid_0's auc: 0.961301
[500]	valid_0's auc: 0.961987
[600]	valid_0's auc: 0.962424
[700]	valid_0's auc: 0.962773
[800]	valid_0's auc: 0.962986
[900]	valid_0's auc: 0.963192
[1000]	valid_0's auc: 0.963377
[1100]	valid_0's auc: 0.963475
[1200]	valid_0's auc: 0.963642
[1300]	valid_0's auc: 0.963747
[1400]	valid_0's auc: 0.963878
[1500]	valid_0's auc: 0.963996
[1600]	valid_0's auc: 0.964019
[1700]	valid_0's auc: 0.964048
[1800]	valid_0's auc: 0.964099
[1900]	valid_0's auc: 0.964159
[2000]	valid_0's auc: 0.964196
[2100]	valid_0's auc: 0.964216
[2200]	valid_0's auc: 0.964237
[2300]	valid_0's auc: 0.964272
[2400]	valid_0's auc: 0.964281
[2500]	valid_0's auc: 0.964297
[2600]	valid_0's auc: 0.96432
[2700]	valid_0's auc: 0.964334
[2800]	valid_0's auc: 0.964345
[2900]	valid_0's auc: 0.964365
[3000]	valid_0's auc: 0.964373
[3100]	valid_0's au

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.945019
[200]	valid_0's auc: 0.95612
[300]	valid_0's auc: 0.959962
[400]	valid_0's auc: 0.961791
[500]	valid_0's auc: 0.962565
[600]	valid_0's auc: 0.962979
[700]	valid_0's auc: 0.963383
[800]	valid_0's auc: 0.963609
[900]	valid_0's auc: 0.963806
[1000]	valid_0's auc: 0.963989
[1100]	valid_0's auc: 0.964196
[1200]	valid_0's auc: 0.964326
[1300]	valid_0's auc: 0.964433
[1400]	valid_0's auc: 0.964546
[1500]	valid_0's auc: 0.964594
[1600]	valid_0's auc: 0.96467
[1700]	valid_0's auc: 0.964752
[1800]	valid_0's auc: 0.964827
[1900]	valid_0's auc: 0.964876
[2000]	valid_0's auc: 0.964898
[2100]	valid_0's auc: 0.964938
[2200]	valid_0's auc: 0.964963
[2300]	valid_0's auc: 0.965004
[2400]	valid_0's auc: 0.965005
[2500]	valid_0's auc: 0.965036
[2600]	valid_0's auc: 0.965056
[2700]	valid_0's auc: 0.96506
[2800]	valid_0's auc: 0.965077
[2900]	valid_0's auc: 0.965085
[3000]	valid_0's auc: 0.965098
[3100]	valid_0's auc

...

Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.946113
[200]	valid_0's auc: 0.956743
[300]	valid_0's auc: 0.960438
[400]	valid_0's auc: 0.962317
[500]	valid_0's auc: 0.963065
[600]	valid_0's auc: 0.963421
[700]	valid_0's auc: 0.963768
[800]	valid_0's auc: 0.964044
[900]	valid_0's auc: 0.964296
[1000]	valid_0's auc: 0.964432
[1100]	valid_0's auc: 0.964568
[1200]	valid_0's auc: 0.96468
[1300]	valid_0's auc: 0.964771
[1400]	valid_0's auc: 0.964849
[1500]	valid_0's auc: 0.964935
[1600]	valid_0's auc: 0.964967
[1700]	valid_0's auc: 0.964997
[1800]	valid_0's auc: 0.965057
[1900]	valid_0's auc: 0.965086
[2000]	valid_0's auc: 0.965108
[2100]	valid_0's auc: 0.965153
[2200]	valid_0's auc: 0.965181
[2300]	valid_0's auc: 0.965213
[2400]	valid_0's auc: 0.965212
[2500]	valid_0's auc: 0.965213
[2600]	valid_0's auc: 0.965227
[2700]	valid_0's auc: 0.965243
[2800]	valid_0's auc: 0.965259
[2900]	valid_0's auc: 0.965261
[3000]	valid_0's auc: 0.965261
[3100]	valid_0's a

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.946667
[200]	valid_0's auc: 0.9576
[300]	valid_0's auc: 0.9613
[400]	valid_0's auc: 0.963026
[500]	valid_0's auc: 0.963747
[600]	valid_0's auc: 0.964218
[700]	valid_0's auc: 0.964457
[800]	valid_0's auc: 0.964678
[900]	valid_0's auc: 0.964849
[1000]	valid_0's auc: 0.965061
[1100]	valid_0's auc: 0.965152
[1200]	valid_0's auc: 0.965245
[1300]	valid_0's auc: 0.965317
[1400]	valid_0's auc: 0.965394
[1500]	valid_0's auc: 0.965491
[1600]	valid_0's auc: 0.965538
[1700]	valid_0's auc: 0.96559
[1800]	valid_0's auc: 0.965651
[1900]	valid_0's auc: 0.965709
[2000]	valid_0's auc: 0.965748
[2100]	valid_0's auc: 0.965806
[2200]	valid_0's auc: 0.965832
[2300]	valid_0's auc: 0.96585
[2400]	valid_0's auc: 0.965865
[2500]	valid_0's auc: 0.965874
[2600]	valid_0's auc: 0.965889
[2700]	valid_0's auc: 0.965886
[2800]	valid_0's auc: 0.965911
[2900]	valid_0's auc: 0.965918
[3000]	valid_0's auc: 0.965909
[3100]	valid_0's auc: 0

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.946028
[200]	valid_0's auc: 0.956131
[300]	valid_0's auc: 0.959984
[400]	valid_0's auc: 0.961763
[500]	valid_0's auc: 0.962673
[600]	valid_0's auc: 0.963109
[700]	valid_0's auc: 0.963409
[800]	valid_0's auc: 0.963708
[900]	valid_0's auc: 0.96403
[1000]	valid_0's auc: 0.964208
[1100]	valid_0's auc: 0.964325
[1200]	valid_0's auc: 0.964409
[1300]	valid_0's auc: 0.964498
[1400]	valid_0's auc: 0.964591
[1500]	valid_0's auc: 0.964647
[1600]	valid_0's auc: 0.964686
[1700]	valid_0's auc: 0.964724
[1800]	valid_0's auc: 0.964782
[1900]	valid_0's auc: 0.964794
[2000]	valid_0's auc: 0.964829
[2100]	valid_0's auc: 0.964869
[2200]	valid_0's auc: 0.9649
[2300]	valid_0's auc: 0.96489
[2400]	valid_0's auc: 0.964916
[2500]	valid_0's auc: 0.964924
[2600]	valid_0's auc: 0.964934
[2700]	valid_0's auc: 0.964939
[2800]	valid_0's auc: 0.964953
[2900]	valid_0's auc: 0.964956
[3000]	valid_0's auc: 0.964954
[3100]	valid_0's auc:

In [87]:
threshold_auc = roc_auc_score(
    y,
    threshold_oof
)

print(
    f"Threshold LightGBM OOF ROC-AUC: "
    f"{threshold_auc:.6f}"
)

Threshold LightGBM OOF ROC-AUC: 0.965130


In [88]:
print("Threshold OOF:", threshold_oof.shape)
print("XGB OOF:", xgb_oof.shape)
print("CatBoost OOF:", cat_oof.shape)
print("LGB OOF:", lgb_oof.shape)

Threshold OOF: (691369,)
XGB OOF: (691369,)
CatBoost OOF: (691369,)
LGB OOF: (691369,)


In [89]:
print(
    "Threshold LGB:",
    roc_auc_score(y, threshold_oof)
)

Threshold LGB: 0.965129806227864


In [90]:
blend_results = []

weights = [
    (0.4, 0.3, 0.3),
    (0.5, 0.25, 0.25),
    (0.5, 0.3, 0.2),
    (0.6, 0.2, 0.2),
    (0.6, 0.25, 0.15),
    (0.7, 0.15, 0.15),
    (0.7, 0.2, 0.1),
    (0.8, 0.1, 0.1),
]

for w_lgb, w_xgb, w_cat in weights:

    pred = (
        w_lgb * threshold_oof
        + w_xgb * xgb_oof
        + w_cat * cat_oof
    )

    auc = roc_auc_score(
        y,
        pred
    )

    blend_results.append({
        "LightGBM": w_lgb,
        "XGBoost": w_xgb,
        "CatBoost": w_cat,
        "ROC-AUC": auc
    })

blend_results = pd.DataFrame(
    blend_results
).sort_values(
    "ROC-AUC",
    ascending=False
)

print(
    blend_results.to_string(index=False)
)

 LightGBM  XGBoost  CatBoost  ROC-AUC
      0.6     0.25      0.15 0.965597
      0.7     0.20      0.10 0.965578
      0.6     0.20      0.20 0.965574
      0.5     0.30      0.20 0.965559
      0.7     0.15      0.15 0.965550
      0.5     0.25      0.25 0.965541
      0.8     0.10      0.10 0.965470
      0.4     0.30      0.30 0.965452


In [91]:
best_blend = blend_results.iloc[0]

print(best_blend)

LightGBM    0.600000
XGBoost     0.250000
CatBoost    0.150000
ROC-AUC     0.965597
Name: 4, dtype: float64


In [92]:
# Current best blend
best_oof_pred = (
    0.60 * threshold_oof
    + 0.25 * xgb_oof
    + 0.15 * cat_oof
)

best_test_pred = (
    0.60 * threshold_test
    + 0.25 * xgb_test
    + 0.15 * cat_test
)

In [93]:
best_auc = roc_auc_score(
    y,
    best_oof_pred
)

print(f"Best Blend OOF ROC-AUC: {best_auc:.6f}")

Best Blend OOF ROC-AUC: 0.965597


In [94]:
submission = pd.DataFrame({
    "id": test["id"],
    "addicted_label": best_test_pred
})

submission.to_csv(
    "submission_best_0965597.csv",
    index=False
)

print(submission.head())
print(submission.shape)

       id  addicted_label
0  691369        0.999457
1  691370        0.943588
2  691371        0.929540
3  691372        0.991903
4  691373        0.998004
(296302, 2)


In [95]:
import os

print(
    os.path.abspath(
        "submission_best_0965597.csv"
    )
)

print(
    submission["addicted_label"].min(),
    submission["addicted_label"].max(),
    submission["addicted_label"].mean()
)

c:\ZZZ D Drive\CRISP\Kaggle Compe\playground-series-s6e8\submission_best_0965597.csv
9.411426930788078e-05 0.9999998825606463 0.710127981317264


In [96]:
X_threshold = X.copy()
X_test_threshold = X_test.copy()

for col, values in thresholds.items():

    for threshold in values:

        safe_name = str(threshold).replace(".", "_")

        X_threshold[f"{col}_gt_{safe_name}"] = (
            X_threshold[col] > threshold
        ).astype("int8")

        X_test_threshold[f"{col}_gt_{safe_name}"] = (
            X_test_threshold[col] > threshold
        ).astype("int8")

        X_threshold[f"{col}_dist_{safe_name}"] = (
            X_threshold[col] - threshold
        )

        X_test_threshold[f"{col}_dist_{safe_name}"] = (
            X_test_threshold[col] - threshold
        )

print("Threshold-only features:", X_threshold.shape[1])

Threshold-only features: 48


In [97]:
threshold_only_oof = np.zeros(len(X_threshold))
threshold_only_test = np.zeros(len(X_test_threshold))

In [98]:
for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print("=" * 60)
    print(f"FOLD {fold}/5")
    print("=" * 60)

    X_train = X_threshold.iloc[train_idx]
    X_valid = X_threshold.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMClassifier(
        **threshold_params
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (X_valid, y_valid)
        ],

        eval_metric="auc",

        categorical_feature=categorical_columns,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=200,
                verbose=True
            ),
            lgb.log_evaluation(
                period=100
            )
        ]
    )

    valid_pred = model.predict_proba(
        X_valid
    )[:, 1]

    test_pred = model.predict_proba(
        X_test_threshold
    )[:, 1]

    threshold_only_oof[valid_idx] = valid_pred

    threshold_only_test += (
        test_pred / len(folds)
    )

    print(
        f"Fold {fold} ROC-AUC: "
        f"{roc_auc_score(y_valid, valid_pred):.6f}"
    )

FOLD 1/5


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.943331
[200]	valid_0's auc: 0.954185
[300]	valid_0's auc: 0.958024
[400]	valid_0's auc: 0.960156
[500]	valid_0's auc: 0.961014
[600]	valid_0's auc: 0.961584
[700]	valid_0's auc: 0.96202
[800]	valid_0's auc: 0.962298
[900]	valid_0's auc: 0.962459
[1000]	valid_0's auc: 0.962677
[1100]	valid_0's auc: 0.962854
[1200]	valid_0's auc: 0.963031
[1300]	valid_0's auc: 0.96317
[1400]	valid_0's auc: 0.963348
[1500]	valid_0's auc: 0.963481
[1600]	valid_0's auc: 0.963582
[1700]	valid_0's auc: 0.963649
[1800]	valid_0's auc: 0.963737
[1900]	valid_0's auc: 0.963797
[2000]	valid_0's auc: 0.963862
[2100]	valid_0's auc: 0.963902
[2200]	valid_0's auc: 0.963955
[2300]	valid_0's auc: 0.964012
[2400]	valid_0's auc: 0.964033
[2500]	valid_0's auc: 0.96404
[2600]	valid_0's auc: 0.964059
[2700]	valid_0's auc: 0.964087
[2800]	valid_0's auc: 0.964102
[2900]	valid_0's auc: 0.964112
[3000]	valid_0's auc: 0.964112
[3100]	valid_0's auc

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.94304
[200]	valid_0's auc: 0.954385
[300]	valid_0's auc: 0.958665
[400]	valid_0's auc: 0.96063
[500]	valid_0's auc: 0.961608
[600]	valid_0's auc: 0.962152
[700]	valid_0's auc: 0.962572
[800]	valid_0's auc: 0.962877
[900]	valid_0's auc: 0.963111
[1000]	valid_0's auc: 0.963289
[1100]	valid_0's auc: 0.963478
[1200]	valid_0's auc: 0.963736
[1300]	valid_0's auc: 0.963919
[1400]	valid_0's auc: 0.964018
[1500]	valid_0's auc: 0.964098
[1600]	valid_0's auc: 0.964171
[1700]	valid_0's auc: 0.964273
[1800]	valid_0's auc: 0.964355
[1900]	valid_0's auc: 0.964396
[2000]	valid_0's auc: 0.964464
[2100]	valid_0's auc: 0.964511
[2200]	valid_0's auc: 0.964552
[2300]	valid_0's auc: 0.964596
[2400]	valid_0's auc: 0.964642
[2500]	valid_0's auc: 0.964684
[2600]	valid_0's auc: 0.964721
[2700]	valid_0's auc: 0.964772
[2800]	valid_0's auc: 0.96482
[2900]	valid_0's auc: 0.964844
[3000]	valid_0's auc: 0.964856
[3100]	valid_0's auc

...

Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.944708
[200]	valid_0's auc: 0.955564
[300]	valid_0's auc: 0.959413
[400]	valid_0's auc: 0.961208
[500]	valid_0's auc: 0.962173
[600]	valid_0's auc: 0.9627
[700]	valid_0's auc: 0.96307
[800]	valid_0's auc: 0.963401
[900]	valid_0's auc: 0.96361
[1000]	valid_0's auc: 0.963792
[1100]	valid_0's auc: 0.964038
[1200]	valid_0's auc: 0.964219
[1300]	valid_0's auc: 0.964345
[1400]	valid_0's auc: 0.964456
[1500]	valid_0's auc: 0.964563
[1600]	valid_0's auc: 0.964647
[1700]	valid_0's auc: 0.964708
[1800]	valid_0's auc: 0.964802
[1900]	valid_0's auc: 0.964843
[2000]	valid_0's auc: 0.964901
[2100]	valid_0's auc: 0.964944
[2200]	valid_0's auc: 0.965003
[2300]	valid_0's auc: 0.965025
[2400]	valid_0's auc: 0.965058
[2500]	valid_0's auc: 0.965089
[2600]	valid_0's auc: 0.965109
[2700]	valid_0's auc: 0.965145
[2800]	valid_0's auc: 0.965175
[2900]	valid_0's auc: 0.965213
[3000]	valid_0's auc: 0.965225
[3100]	valid_0's auc:

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.944601
[200]	valid_0's auc: 0.955887
[300]	valid_0's auc: 0.959942
[400]	valid_0's auc: 0.961881
[500]	valid_0's auc: 0.962884
[600]	valid_0's auc: 0.963394
[700]	valid_0's auc: 0.963859
[800]	valid_0's auc: 0.964165
[900]	valid_0's auc: 0.964337
[1000]	valid_0's auc: 0.964507
[1100]	valid_0's auc: 0.9647
[1200]	valid_0's auc: 0.964847
[1300]	valid_0's auc: 0.964993
[1400]	valid_0's auc: 0.965084
[1500]	valid_0's auc: 0.96521
[1600]	valid_0's auc: 0.965326
[1700]	valid_0's auc: 0.9654
[1800]	valid_0's auc: 0.965463
[1900]	valid_0's auc: 0.965504
[2000]	valid_0's auc: 0.965539
[2100]	valid_0's auc: 0.965583
[2200]	valid_0's auc: 0.965628
[2300]	valid_0's auc: 0.965652
[2400]	valid_0's auc: 0.965676
[2500]	valid_0's auc: 0.96569
[2600]	valid_0's auc: 0.965692
[2700]	valid_0's auc: 0.965714
[2800]	valid_0's auc: 0.965719
[2900]	valid_0's auc: 0.965727
[3000]	valid_0's auc: 0.96574
[3100]	valid_0's auc: 0.

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.944467
[200]	valid_0's auc: 0.954612
[300]	valid_0's auc: 0.958804
[400]	valid_0's auc: 0.960644
[500]	valid_0's auc: 0.961689
[600]	valid_0's auc: 0.962292
[700]	valid_0's auc: 0.962651
[800]	valid_0's auc: 0.963007
[900]	valid_0's auc: 0.963277
[1000]	valid_0's auc: 0.963416
[1100]	valid_0's auc: 0.963612
[1200]	valid_0's auc: 0.963782
[1300]	valid_0's auc: 0.96394
[1400]	valid_0's auc: 0.964009
[1500]	valid_0's auc: 0.964122
[1600]	valid_0's auc: 0.964242
[1700]	valid_0's auc: 0.964371
[1800]	valid_0's auc: 0.964415
[1900]	valid_0's auc: 0.96446
[2000]	valid_0's auc: 0.964505
[2100]	valid_0's auc: 0.964547
[2200]	valid_0's auc: 0.964607
[2300]	valid_0's auc: 0.964635
[2400]	valid_0's auc: 0.964664
[2500]	valid_0's auc: 0.964669
[2600]	valid_0's auc: 0.964702
[2700]	valid_0's auc: 0.964731
[2800]	valid_0's auc: 0.964738
[2900]	valid_0's auc: 0.964754
[3000]	valid_0's auc: 0.96477
[3100]	valid_0's auc

In [99]:
threshold_only_auc = roc_auc_score(
    y,
    threshold_only_oof
)

print(
    f"Threshold-only LightGBM OOF ROC-AUC: "
    f"{threshold_only_auc:.6f}"
)

Threshold-only LightGBM OOF ROC-AUC: 0.965005


In [100]:
blend_results_fine = []

for w_lgb in np.arange(0.50, 0.91, 0.01):

    for w_xgb in np.arange(0.05, 0.46, 0.01):

        w_cat = 1.0 - w_lgb - w_xgb

        # CatBoost must have at least 5% weight
        if w_cat < 0.05:
            continue

        pred = (
            w_lgb * threshold_oof
            + w_xgb * xgb_oof
            + w_cat * cat_oof
        )

        auc = roc_auc_score(
            y,
            pred
        )

        blend_results_fine.append({
            "LightGBM": w_lgb,
            "XGBoost": w_xgb,
            "CatBoost": w_cat,
            "ROC-AUC": auc
        })

blend_results_fine = pd.DataFrame(
    blend_results_fine
).sort_values(
    "ROC-AUC",
    ascending=False
).reset_index(drop=True)

print(
    blend_results_fine.head(20).to_string(index=False)
)

 LightGBM  XGBoost  CatBoost  ROC-AUC
     0.62     0.27      0.11 0.965602
     0.62     0.28      0.10 0.965602
     0.63     0.27      0.10 0.965602
     0.61     0.28      0.11 0.965602
     0.61     0.27      0.12 0.965601
     0.63     0.28      0.09 0.965601
     0.63     0.26      0.11 0.965601
     0.62     0.26      0.12 0.965601
     0.61     0.29      0.10 0.965601
     0.64     0.27      0.09 0.965601
     0.62     0.29      0.09 0.965601
     0.64     0.26      0.10 0.965601
     0.60     0.28      0.12 0.965601
     0.60     0.29      0.11 0.965600
     0.64     0.28      0.08 0.965600
     0.61     0.26      0.13 0.965600
     0.63     0.29      0.08 0.965600
     0.63     0.25      0.12 0.965600
     0.60     0.27      0.13 0.965600
     0.64     0.25      0.11 0.965600


In [101]:
best_weights = blend_results_fine.iloc[0]

print("BEST BLEND")
print("=" * 40)
print(f"LightGBM : {best_weights['LightGBM']:.2f}")
print(f"XGBoost  : {best_weights['XGBoost']:.2f}")
print(f"CatBoost : {best_weights['CatBoost']:.2f}")
print(f"ROC-AUC  : {best_weights['ROC-AUC']:.6f}")

BEST BLEND
LightGBM : 0.62
XGBoost  : 0.27
CatBoost : 0.11
ROC-AUC  : 0.965602


In [102]:
print("Previous best : 0.965597")
print(
    f"New best      : "
    f"{best_weights['ROC-AUC']:.6f}"
)

print(
    f"Improvement   : "
    f"{best_weights['ROC-AUC'] - 0.965597:+.6f}"
)

Previous best : 0.965597
New best      : 0.965602
Improvement   : +0.000005


In [103]:
best_oof_pred_final = (
    0.62 * threshold_oof
    + 0.27 * xgb_oof
    + 0.11 * cat_oof
)

print(
    f"Final OOF ROC-AUC: "
    f"{roc_auc_score(y, best_oof_pred_final):.6f}"
)

Final OOF ROC-AUC: 0.965602


In [104]:
best_test_pred_final = (
    0.62 * threshold_test
    + 0.27 * xgb_test
    + 0.11 * cat_test
)

In [105]:
submission_final = pd.DataFrame({
    "id": test["id"],
    "addicted_label": best_test_pred_final
})

submission_final.to_csv(
    "submission_final_0965602.csv",
    index=False
)

print(submission_final.head())
print()
print("Shape:", submission_final.shape)

       id  addicted_label
0  691369        0.999465
1  691370        0.943620
2  691371        0.928830
3  691372        0.992066
4  691373        0.998041

Shape: (296302, 2)


In [106]:
print(
    "Min:",
    submission_final["addicted_label"].min()
)

print(
    "Max:",
    submission_final["addicted_label"].max()
)

print(
    "Mean:",
    submission_final["addicted_label"].mean()
)

print(
    "Missing:",
    submission_final["addicted_label"].isna().sum()
)

Min: 9.407895820164794e-05
Max: 0.9999998963433037
Mean: 0.710166932384274
Missing: 0
